In [1]:
! python3 -m pip show opencv-python mediapipe numpy

Name: opencv-python
Version: 4.11.0.86
Summary: Wrapper package for OpenCV python bindings.
Home-page: https://github.com/opencv/opencv-python
Author: 
Author-email: 
License: Apache 2.0
Location: /opt/anaconda3/envs/marvel-magic/lib/python3.11/site-packages
Requires: numpy
Required-by: 
---
Name: mediapipe
Version: 0.10.21
Summary: MediaPipe is the simplest way for researchers and developers to build world-class ML solutions and applications for mobile, edge, cloud and the web.
Home-page: https://github.com/google/mediapipe
Author: The MediaPipe Authors
Author-email: mediapipe@google.com
License: Apache 2.0
Location: /opt/anaconda3/envs/marvel-magic/lib/python3.11/site-packages
Requires: absl-py, attrs, flatbuffers, jax, jaxlib, matplotlib, numpy, opencv-contrib-python, protobuf, sentencepiece, sounddevice
Required-by: 
---
Name: numpy
Version: 1.26.4
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email:

In [2]:
import cv2
import time
import math
import mediapipe as mp
import numpy as np

In [3]:
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)

OpenCV: 4.11.0
MediaPipe: 0.10.21


In [4]:
class HandTracker:
    def __init__(
        self,
        max_hands=2,
        detection_confidence=0.6,
        tracking_confidence=0.6
    ):
        self.mp_hands = mp.solutions.hands
        self.mp_draw = mp.solutions.drawing_utils

        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=max_hands,
            min_detection_confidence=detection_confidence,
            min_tracking_confidence=tracking_confidence
        )
        self.results = None

    def find_hands(self, frame, draw=True):
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        self.results = self.hands.process(rgb_frame)

        if draw and self.results.multi_hand_landmarks:
            for hand_landmarks in self.results.multi_hand_landmarks:
                self.mp_draw.draw_landmarks(
                    frame,
                    hand_landmarks,
                    self.mp_hands.HAND_CONNECTIONS
                )

        return frame

    def get_landmarks(self, frame_width, frame_height):
        all_hands = []

        if self.results and self.results.multi_hand_landmarks:
            for hand_landmarks in self.results.multi_hand_landmarks:
                hand_points = []

                for landmark in hand_landmarks.landmark:
                    x = int(landmark.x * frame_width)
                    y = int(landmark.y * frame_height)
                    z = landmark.z
                    hand_points.append((x, y, z))

                all_hands.append(hand_points)

        return all_hands

    def close(self):
        self.hands.close()


In [5]:
def run_day1_hand_tracking(camera_index=0):
    cap = cv2.VideoCapture(camera_index)

    if not cap.isOpened():
        print("ERROR: Cannot access webcam. Check camera permissions or try camera_index=1.")
        return

    tracker = HandTracker(max_hands=2)
    previous_time = 0

    print("MYSTIC MOTION — DAY 1")
    print("Press Q in the camera window to exit.")

    try:
        while True:
            success, frame = cap.read()
            if not success:
                print("Failed to capture a frame.")
                break

            # Mirror the image for natural interaction.
            frame = cv2.flip(frame, 1)
            height, width, _ = frame.shape

            frame = tracker.find_hands(frame)
            hands = tracker.get_landmarks(width, height)

            cv2.putText(
                frame,
                f"Hands detected: {len(hands)}",
                (20, 45),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 100),
                2
            )

            for i, hand in enumerate(hands):
                cv2.putText(
                    frame,
                    f"Hand {i + 1}: {len(hand)} landmarks",
                    (20, 85 + i * 35),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,
                    (255, 255, 0),
                    2
                )

            current_time = time.time()
            fps = 1 / (current_time - previous_time) if previous_time else 0
            previous_time = current_time

            cv2.putText(
                frame,
                f"FPS: {int(fps)}",
                (width - 140, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 255),
                2
            )

            cv2.imshow("MYSTIC MOTION — Day 1 Hand Tracking", frame)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    finally:
        tracker.close()
        cap.release()
        cv2.destroyAllWindows()


run_day1_hand_tracking(camera_index=0)

MYSTIC MOTION — DAY 1
Press Q in the camera window to exit.


I0000 00:00:1793862962.270653  779222 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1793862962.281251  779364 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793862962.288436  779364 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793862962.674915  779364 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


In [6]:
class GestureRecognizer:
    @staticmethod
    def distance(point1, point2):
        return math.sqrt(
            (point1[0] - point2[0]) ** 2 +
            (point1[1] - point2[1]) ** 2
        )

    def recognize_gesture(self, landmarks):
        if len(landmarks) != 21:
            return "UNKNOWN"

        finger_tips = [8, 12, 16, 20]
        finger_pips = [6, 10, 14, 18]

        fingers_up = [
            landmarks[tip][1] < landmarks[pip][1]
            for tip, pip in zip(finger_tips, finger_pips)
        ]

        thumb_tip = landmarks[4]
        index_mcp = landmarks[5]

        thumb_distance = self.distance(thumb_tip, index_mcp)
        pinch_distance = self.distance(landmarks[4], landmarks[8])

        hand_size = self.distance(landmarks[0], landmarks[9])
        if hand_size == 0:
            return "UNKNOWN"

        pinch_ratio = pinch_distance / hand_size
        thumb_ratio = thumb_distance / hand_size

        if pinch_ratio < 0.35:
            return "PINCH"

        if all(fingers_up) and thumb_ratio > 0.45:
            return "OPEN PALM"

        if not any(fingers_up) and thumb_ratio < 0.45:
            return "CLOSED FIST"

        if (
            fingers_up[0]
            and fingers_up[1]
            and not fingers_up[2]
            and not fingers_up[3]
        ):
            return "TWO FINGERS"

        return "UNKNOWN"

In [7]:
def run_day2_gesture_recognition(camera_index=0):
    cap = cv2.VideoCapture(camera_index)

    if not cap.isOpened():
        print("Cannot access webcam. Check camera permissions or try camera_index=1.")
        return

    tracker = HandTracker(max_hands=2)
    recognizer = GestureRecognizer()

    print("MYSTIC MOTION — DAY 2")
    print("Show the four starter gestures. Press Q in the camera window to exit.")

    try:
        while True:
            success, frame = cap.read()
            if not success:
                print("Failed to capture a frame.")
                break

            frame = cv2.flip(frame, 1)
            height, width, _ = frame.shape

            frame = tracker.find_hands(frame)
            hands = tracker.get_landmarks(width, height)

            for index, hand in enumerate(hands):
                gesture = recognizer.recognize_gesture(hand)

                cv2.putText(
                    frame,
                    gesture,
                    (20, 65 + index * 55),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.9,
                    (0, 255, 0),
                    2
                )

                cv2.circle(
                    frame,
                    (width - 45, 45 + index * 50),
                    12,
                    (0, 255, 0),
                    -1
                )

            cv2.putText(
                frame,
                "MYSTIC MOTION | GESTURE ENGINE",
                (20, height - 25),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (255, 255, 255),
                2
            )

            cv2.imshow("MYSTIC MOTION — Day 2 Gesture Recognition", frame)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    finally:
        tracker.close()
        cap.release()
        cv2.destroyAllWindows()


run_day2_gesture_recognition(camera_index=0)

MYSTIC MOTION — DAY 2
Show the four starter gestures. Press Q in the camera window to exit.


I0000 00:00:1793862970.511038  779222 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3
W0000 00:00:1793862970.519316  779508 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793862970.526117  779508 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [8]:
# python -m pip install --force-reinstall --no-cache-dir --no-deps pandas==2.2.3

In [9]:
import sys
from turtle import pd
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

assert sys.version_info >= (3, 10), "Use the Python 3.11 marvel-magic environment."
assert hasattr(mp, "solutions"), (
    "This notebook expects MediaPipe's legacy Hands API. "
    "Check that MediaPipe 0.10.21 is installed in the selected kernel."
)

DATA_DIR = Path("mystic_motion_data")
DATA_DIR.mkdir(exist_ok=True)

RAW_CSV = DATA_DIR / "gesture_landmarks_raw.csv"
PROCESSED_CSV = DATA_DIR / "gesture_landmarks_processed.csv"

GESTURES = {
    "1": "PINCH",
    "2": "OPEN PALM",
    "3": "CLOSED FIST",
    "4": "TWO FINGERS",
}
FEATURE_COLUMNS = [
    f"{axis}_{i}" for i in range(21) for axis in ("x", "y", "z")
]
print("\nGesture keys:", GESTURES)
print("Data folder:", DATA_DIR.resolve())

Python: 3.11.18
Interpreter: /opt/anaconda3/envs/marvel-magic/bin/python
OpenCV: 4.11.0
MediaPipe: 0.10.21
NumPy: 1.26.4
Pandas: 2.2.3

Gesture keys: {'1': 'PINCH', '2': 'OPEN PALM', '3': 'CLOSED FIST', '4': 'TWO FINGERS'}
Data folder: /Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/mystic_motion_data


In [ ]:
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

def collect_gesture_samples():
    rows = []
    selected_gesture = None
    camera = cv2.VideoCapture(0)

    if not camera.isOpened():
        raise RuntimeError(
            "Could not open webcam. Check macOS Camera permission for VS Code "
            "(or the app hosting Jupyter), then close other apps using the camera."
        )

    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        model_complexity=1,
        min_detection_confidence=0.65,
        min_tracking_confidence=0.60,
    )

    print("Select gesture 1–4, press SPACE to capture, Q to save and quit.")
    try:
        while True:
            ok, frame = camera.read()
            if not ok:
                print("Camera frame could not be read.")
                break

            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = hands.process(rgb)

            detected_landmarks = None
            handedness = "Unknown"

            if result.multi_hand_landmarks:
                hand_lms = result.multi_hand_landmarks[0]
                detected_landmarks = [
                    (lm.x, lm.y, lm.z) for lm in hand_lms.landmark
                ]
                if result.multi_handedness:
                    handedness = result.multi_handedness[0].classification[0].label
                mp_draw.draw_landmarks(
                    frame, hand_lms, mp_hands.HAND_CONNECTIONS
                )

            counts = Counter(row["label"] for row in rows)
            cv2.rectangle(frame, (0, 0), (frame.shape[1], 115), (20, 20, 20), -1)
            status = selected_gesture if selected_gesture else "Select gesture (1-4)"
            cv2.putText(frame, f"Selected: {status}", (12, 28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            cv2.putText(frame, "SPACE: capture | Q: save & quit", (12, 57),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
            count_text = "  ".join(
                f"{name}: {counts.get(name, 0)}"
                for name in GESTURES.values()
            )
            cv2.putText(frame, count_text, (12, 88),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.48, (100, 255, 100), 1)
            cv2.putText(frame, "HAND DETECTED" if detected_landmarks else "NO HAND",
                        (12, frame.shape[0] - 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                        (0, 255, 0) if detected_landmarks else (0, 0, 255), 2)

            cv2.imshow("MYSTIC MOTION — Day 3 Dataset Collector", frame)
            key = cv2.waitKey(1) & 0xFF

            if key == ord("q"):
                break
            elif chr(key) in GESTURES if key < 128 else False:
                selected_gesture = GESTURES[chr(key)]
                print("Selected:", selected_gesture)
            elif key == 32:  # SPACE
                if selected_gesture is None:
                    print("Choose a gesture first using keys 1–4.")
                elif detected_landmarks is None:
                    print("No hand detected — sample skipped.")
                else:
                    row = {
                        "label": selected_gesture,
                        "handedness": handedness,
                    }
                    for i, (x, y, z) in enumerate(detected_landmarks):
                        row[f"x_{i}"] = x
                        row[f"y_{i}"] = y
                        row[f"z_{i}"] = z
                    rows.append(row)
                    print(f"Captured {selected_gesture}: "
                          f"{sum(r['label'] == selected_gesture for r in rows)}")

    finally:
        camera.release()
        hands.close()
        cv2.destroyAllWindows()

    if not rows:
        print("No samples captured. Run this cell again and capture samples.")
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    ordered = ["label", "handedness"] + FEATURE_COLUMNS
    df = df[ordered]
    df.to_csv(RAW_CSV, index=False)
    print(f"\nSaved {len(df)} samples to: {RAW_CSV.resolve()}")
    print("\nSamples per gesture:")
    print(df["label"].value_counts())
    return df

raw_df = collect_gesture_samples()

Select gesture 1–4, press SPACE to capture, Q to save and quit.


I0000 00:00:1793862983.793931  779222 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3
W0000 00:00:1793862983.801857  779709 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793862983.815001  779709 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


No samples captured. Run this cell again and capture samples.


In [12]:
if not RAW_CSV.exists():
    raise FileNotFoundError(
        f"{RAW_CSV} not found. Complete Day 3 collection first."
    )

raw_df = pd.read_csv(RAW_CSV)
required = {"label", "handedness", *FEATURE_COLUMNS}
missing = required.difference(raw_df.columns)
if missing:
    raise ValueError(f"Dataset is missing columns: {sorted(missing)}")

print("Dataset shape:", raw_df.shape)
print("\nClass counts:")
display(raw_df["label"].value_counts().rename_axis("gesture").to_frame("samples"))

if raw_df["label"].nunique() < 2:
    raise ValueError("Collect at least two gesture classes before preprocessing.")
if raw_df[FEATURE_COLUMNS].isna().any().any():
    raise ValueError("Missing landmark values found. Recollect clean samples.")

FileNotFoundError: mystic_motion_data/gesture_landmarks_raw.csv not found. Complete Day 3 collection first.

In [ ]:
# Cell 4 — Normalize each 21-point hand into 63 features
def normalize_landmarks(row):
    points = row[FEATURE_COLUMNS].to_numpy(dtype=np.float32).reshape(21, 3)

    # Translation invariance: wrist (landmark 0) becomes (0, 0, 0).
    points = points - points[0]

    # Scale invariance: use the maximum 2D distance from the wrist.
    xy_distances = np.linalg.norm(points[:, :2], axis=1)
    scale = float(np.max(xy_distances))

    if not np.isfinite(scale) or scale < 1e-6:
        raise ValueError("Degenerate hand landmarks detected; recollect this sample.")

    points = points / scale
    return points.reshape(-1)

X = np.vstack([normalize_landmarks(row) for _, row in raw_df.iterrows()])
y_text = raw_df["label"].astype(str).to_numpy()

print("Feature matrix:", X.shape)
print("Labels:", y_text.shape)
print("Features per sample:", X.shape[1])
print("All finite:", bool(np.isfinite(X).all()))
assert X.shape[1] == 63
assert np.isfinite(X).all()

NameError: name 'raw_df' is not defined

In [ ]:
# Cell 5 — Encode labels and create a stratified train/test split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
label_names = list(label_encoder.classes_)

# Stratification preserves each gesture's proportion in both splits.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Label mapping:")
for index, name in enumerate(label_names):
    print(f"  {name}: {index}")

print("\nTrain features:", X_train.shape)
print("Test features:", X_test.shape)
print("Train labels:", y_train.shape)
print("Test labels:", y_test.shape)

processed_df = pd.DataFrame(X, columns=FEATURE_COLUMNS)
processed_df.insert(0, "label_id", y)
processed_df.insert(0, "label", y_text)
processed_df.to_csv(PROCESSED_CSV, index=False)

print("\nSaved normalized dataset:", PROCESSED_CSV.resolve())
print("Training and test split completed (80% / 20%).")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Cell 6 — Final Day 3 & 4 quality report
print("MYSTIC MOTION — DAY 3 & 4 REPORT")
print("=" * 42)
print("Raw samples:", len(raw_df))
print("Gesture classes:", len(label_names))
print("Features per sample:", X.shape[1])
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Normalized values finite:", np.isfinite(X).all())
print("\nClass distribution:")
print(raw_df["label"].value_counts().to_string())
print("\nGenerated files:")
print("1.", RAW_CSV.resolve())
print("2.", PROCESSED_CSV.resolve())

print("\nNEXT: Day 5 — Train and compare gesture classifiers.")
print("Note: dynamic gestures (circle, swipe, launch) need time-sequence data; "
      "these static landmark rows are not enough for those actions.")

MYSTIC MOTION — DAY 3 & 4 REPORT


NameError: name 'raw_df' is not defined